### Task A: Full Lab Manual — Knowledge Representation & Reasoning
#### Python: University IT Support Expert System

In [1]:
def forward_chaining(facts, rules):
    new_fact_added = True
    while new_fact_added:
        new_fact_added = False
        for rule in rules:
            if all(condition in facts for condition in rule["conditions"]):
                conclusion = rule["conclusion"]
                if conclusion not in facts:
                    facts.add(conclusion)
                    print("New fact derived:", conclusion)
                    new_fact_added = True
    return facts

laptop_facts = {"wifi_connected", "internet_not_working",
                "other_devices_online", "invalid_ip"}
laptop_rules = [
    {"conditions": ["wifi_connected", "internet_not_working"],
     "conclusion": "wifi_without_internet"},
    {"conditions": ["wifi_without_internet", "other_devices_online"],
     "conclusion": "device_specific_problem"},
    {"conditions": ["device_specific_problem", "invalid_ip"],
     "conclusion": "ip_configuration_issue"}
]
laptop_recommendations = {
    "ip_configuration_issue": "Renew your IP address and reconnect to the network.",
    "device_specific_problem": "Check the network settings of your laptop.",
    "wifi_without_internet": "Check router and internet connectivity."
}

def display_result(facts, rules, recommendations):
    initial = facts.copy()
    print("Initial facts:", ", ".join(sorted(initial)))
    result = forward_chaining(facts.copy(), rules)
    print("Newly derived facts:", ", ".join(sorted(result - initial)) or "None")
    print("Final Knowledge Base:", ", ".join(sorted(result)))
    print("Possible Diagnoses and Solutions:")
    found = False
    for diagnosis, solution in recommendations.items():
        if diagnosis in result:
            print("Diagnosis:", diagnosis)
            print("Recommended Solution:", solution)
            found = True
    if not found:
        print("No supported diagnosis found.")
    return result

result = display_result(laptop_facts, laptop_rules, laptop_recommendations)

Initial facts: internet_not_working, invalid_ip, other_devices_online, wifi_connected
New fact derived: wifi_without_internet
New fact derived: device_specific_problem
New fact derived: ip_configuration_issue
Newly derived facts: device_specific_problem, ip_configuration_issue, wifi_without_internet
Final Knowledge Base: device_specific_problem, internet_not_working, invalid_ip, ip_configuration_issue, other_devices_online, wifi_connected, wifi_without_internet
Possible Diagnoses and Solutions:
Diagnosis: ip_configuration_issue
Recommended Solution: Renew your IP address and reconnect to the network.
Diagnosis: device_specific_problem
Recommended Solution: Check the network settings of your laptop.
Diagnosis: wifi_without_internet
Recommended Solution: Check router and internet connectivity.


In [2]:
def ask_yes_no(prompt):
    return input(prompt + " (yes/no): ").strip().lower() == "yes"

def laptop_support():
    facts = set()
    if ask_yes_no("Is Wi-Fi connected?"):
        facts.add("wifi_connected")
    if not ask_yes_no("Is internet working?"):
        facts.add("internet_not_working")
    if ask_yes_no("Are other devices online?"):
        facts.add("other_devices_online")
    if ask_yes_no("Is IP configuration invalid?"):
        facts.add("invalid_ip")
    return display_result(facts, laptop_rules, laptop_recommendations)

from unittest.mock import patch

def demonstrate_inputs(function, answers):
    answers = iter(answers)
    def supplied_input(prompt):
        answer = next(answers)
        print(prompt + answer)
        return answer
    with patch("builtins.input", supplied_input):
        return function()

print("Recorded input demonstration: laptop IP issue")
result = demonstrate_inputs(laptop_support, ["yes", "no", "yes", "yes"])

Recorded input demonstration: laptop IP issue
Is Wi-Fi connected? (yes/no): yes
Is internet working? (yes/no): no
Are other devices online? (yes/no): yes
Is IP configuration invalid? (yes/no): yes
Initial facts: internet_not_working, invalid_ip, other_devices_online, wifi_connected
New fact derived: wifi_without_internet
New fact derived: device_specific_problem
New fact derived: ip_configuration_issue
Newly derived facts: device_specific_problem, ip_configuration_issue, wifi_without_internet
Final Knowledge Base: device_specific_problem, internet_not_working, invalid_ip, ip_configuration_issue, other_devices_online, wifi_connected, wifi_without_internet
Possible Diagnoses and Solutions:
Diagnosis: ip_configuration_issue
Recommended Solution: Renew your IP address and reconnect to the network.
Diagnosis: device_specific_problem
Recommended Solution: Check the network settings of your laptop.
Diagnosis: wifi_without_internet
Recommended Solution: Check router and internet connectivity.


The initial facts derive wifi_without_internet, device_specific_problem and ip_configuration_issue. The engine applies rules until no new facts remain. Recorded answers exercise the input() calls; call laptop_support() or printer_support() directly for live input.

#### Prolog: Facts, rules, recursion, social networks, first-order logic and puzzles

In [3]:
import os
import shutil
import subprocess

SWIPL = shutil.which("swipl") or os.environ.get("SWIPL")
if SWIPL is None:
    raise RuntimeError("Install SWI-Prolog and make swipl available on PATH.")

def run_prolog(filename, queries):
    completed = subprocess.run(
        [SWIPL, "-q", "-s", "query_runner.pl", "--", filename, *queries],
        text=True, capture_output=True, check=True
    )
    print(completed.stdout, end="")
    if completed.stderr:
        print(completed.stderr, end="")

In [4]:
%%writefile query_runner.pl
:- initialization(main, main).
main([File|Queries]) :-
    consult(File), maplist(show_query, Queries).
show_query(Text) :-
    format('?- ~w~n', [Text]),
    read_term_from_atom(Text, Goal, [variable_names(Vars)]),
    findall(Vars, Goal, Answers),
    ( Answers == [] -> writeln('false.')
    ; Vars == [] -> writeln('true.')
    ; maplist(show_answer, Answers)
    ), nl.
show_answer(Vars) :-
    write_term(Vars, [quoted(true)]), writeln('.').

Overwriting query_runner.pl


In [5]:
%%writefile manual_family.pl
:- multifile parent/2, female/1.
parent(tom, bob).
parent(tom, liz).
parent(bob, ann).
parent(bob, pat).
parent(pat, jim).
male(tom). male(bob). male(jim).
female(liz). female(ann). female(pat).
father(X, Y) :- parent(X, Y), male(X).
mother(X, Y) :- parent(X, Y), female(X).
grandparent(X, Z) :- parent(X, Y), parent(Y, Z).
sibling(X, Y) :- parent(P, X), parent(P, Y), X \== Y.
ancestor(X, Y) :- parent(X, Y).
ancestor(X, Y) :- parent(X, Z), ancestor(Z, Y).


Overwriting manual_family.pl


In [6]:
run_prolog('manual_family.pl', [
    'parent(tom, bob).',
    'parent(tom, X).',
    'parent(X, ann).',
    'parent(tom, X), parent(X, Y).',
    'parent(ann, X).',
    'parent(X, Y).',
    'female(X).',
    'male(X), female(X).',
    'parent(liz, X).',
    'father(tom, bob).',
    'mother(pat, jim).',
    'grandparent(tom, Who).',
    'sibling(ann, X).',
    'ancestor(tom, X).',
    'ancestor(ann, tom).',
    'listing(parent).',
])

?- parent(tom, bob).
true.

?- parent(tom, X).
['X'=bob].
['X'=liz].

?- parent(X, ann).
['X'=bob].

?- parent(tom, X), parent(X, Y).
['X'=bob,'Y'=ann].
['X'=bob,'Y'=pat].

?- parent(ann, X).
false.

?- parent(X, Y).
['X'=tom,'Y'=bob].
['X'=tom,'Y'=liz].
['X'=bob,'Y'=ann].
['X'=bob,'Y'=pat].
['X'=pat,'Y'=jim].

?- female(X).
['X'=liz].
['X'=ann].
['X'=pat].

?- male(X), female(X).
false.

?- parent(liz, X).
false.

?- father(tom, bob).
true.

?- mother(pat, jim).
true.

?- grandparent(tom, Who).
['Who'=ann].
['Who'=pat].

?- sibling(ann, X).
['X'=pat].

?- ancestor(tom, X).
['X'=bob].
['X'=liz].
['X'=ann].
['X'=pat].
['X'=jim].

?- ancestor(ann, tom).
false.

?- listing(parent).
:- multifile parent/2.

parent(tom, bob).
parent(tom, liz).
parent(bob, ann).
parent(bob, pat).
parent(pat, jim).

true.



In [7]:
%%writefile manual_family_extension.pl
:- consult('manual_family.pl').
:- discontiguous parent/2.
:- discontiguous female/1.
parent(jim, zoe).
female(zoe).

Overwriting manual_family_extension.pl


In [8]:
run_prolog('manual_family_extension.pl', [
    'parent(Who, zoe).',
    'ancestor(tom, zoe).',
])

?- parent(Who, zoe).
['Who'=jim].

?- ancestor(tom, zoe).
true.



`parent(liz, X)` fails because no child of Liz is recorded; it does not establish that Liz is childless. Prolog unifies variables with matching facts and backtracks to find further solutions. The recursive ancestor rule follows a parent edge and has a direct-parent base case. This family graph is acyclic.

In [9]:
%%writefile manual_social.pl
friend(ali, sara).
friend(sara, omar).
friend(sara, hina).
friend(omar, zain).
friend(ali, omar).
knows(X, Y) :- friend(X, Y) ; friend(Y, X).
may_know(X, Z) :- knows(X, Y), knows(Y, Z), X \== Z, \+ knows(X, Z).
without_self_guard(X, Z) :- knows(X, Y), knows(Y, Z), \+ knows(X, Z).
without_known_guard(X, Z) :- knows(X, Y), knows(Y, Z), X \== Z.
sibling(ali, hina).
sibling(hina, ali).
:- table connected_friend/2.
connected_friend(X, Y) :- knows(X, Y).
connected_friend(X, Z) :- connected_friend(X, Y), connected_friend(Y, Z).

Overwriting manual_social.pl


In [10]:
run_prolog('manual_social.pl', [
    'may_know(ali, X).',
    'setof(X, without_self_guard(ali, X), People).',
    'setof(X, without_known_guard(ali, X), People).',
    'sibling(ali, X).',
    'setof(X, connected_friend(ali, X), People).',
])

?- may_know(ali, X).
['X'=hina].
['X'=zain].

?- setof(X, without_self_guard(ali, X), People).
['X'=_11320,'People'=[ali,hina,zain]].

?- setof(X, without_known_guard(ali, X), People).
['X'=_11470,'People'=[hina,omar,sara,zain]].

?- sibling(ali, X).
['X'=hina].

?- setof(X, connected_friend(ali, X), People).
['X'=_11726,'People'=[ali,hina,omar,sara,zain]].



Removing the self guard admits Ali; removing the known-person guard admits existing friends Omar and Sara. A recursive friends-of-friends rule can revisit cycles indefinitely under ordinary depth-first execution. Here tabling prevents repeated evaluation; unlike the acyclic parent example, the symmetric friendship graph has cycles.

In [11]:
%%writefile manual_logic.pl
person(ali). person(sara). person(omar). person(hina).
student(ali). student(sara). student(omar).
teacher(hina).
studies(ali). studies(sara). studies(omar).
likes(ali, ai). likes(sara, ml). likes(sara, ai).
cat(tom). cat(kitty). black(kitty).
income(ali, 80000). income(sara, 40000). income(omar, 90000).
defaulted(omar).
approve(C) :- income(C, I), I >= 50000, \+ defaulted(C).
order_total(o1, 30). order_total(o2, 75). order_total(o3, 120).
free_delivery(O) :- order_total(O, T), T > 50.
animal(X) :- cat(X).

Overwriting manual_logic.pl


In [12]:
run_prolog('manual_logic.pl', [
    'forall(student(X), studies(X)).',
    'student(X), likes(X, ai).',
    '\\+ (student(X), \\+ studies(X)).',
    'cat(X), black(X).',
    'forall(person(X), (student(X), studies(X))).',
    'forall(student(S), likes(S, _)).',
    'student(S), \\+ likes(S, _).',
    'approve(C).',
    'free_delivery(O).',
    'findall(S, likes(S, ai), L).',
    'aggregate_all(count, student(_), N).',
    'owns(X, car) = owns(sara, Y).',
    'play(Song, Artist) = play(imagine, john_lennon).',
    'studies(hina).',
    'forall(cat(X), animal(X)).',
    '\\+ (student(X), teacher(X)).',
    'forall((student(S), studies(S)), likes(S, _)).',
    'student(S), studies(S), \\+ likes(S, _).',
])

?- forall(student(X), studies(X)).
['X'=_11336].

?- student(X), likes(X, ai).
['X'=ali].
['X'=sara].

?- \+ (student(X), \+ studies(X)).
['X'=_11590].

?- cat(X), black(X).
['X'=kitty].

?- forall(person(X), (student(X), studies(X))).
false.

?- forall(student(S), likes(S, _)).
false.

?- student(S), \+ likes(S, _).
['S'=omar].

?- approve(C).
['C'=ali].

?- free_delivery(O).
['O'=o2].
['O'=o3].

?- findall(S, likes(S, ai), L).
['S'=_12316,'L'=[ali,sara]].

?- aggregate_all(count, student(_), N).
['N'=3].

?- owns(X, car) = owns(sara, Y).
['X'=sara,'Y'=car].

?- play(Song, Artist) = play(imagine, john_lennon).
['Song'=imagine,'Artist'=john_lennon].

?- studies(hina).
false.

?- forall(cat(X), animal(X)).
['X'=_14864].

?- \+ (student(X), teacher(X)).
['X'=_14976].

?- forall((student(S), studies(S)), likes(S, _)).
false.

?- student(S), studies(S), \+ likes(S, _).
['S'=omar].



Every student studies, but not every person is a studying student: Hina is a teacher. Omar is the counterexample to every student liking some subject. Negation means failure to prove from the current knowledge base; `studies(hina)` is not recorded. Loan approval succeeds only for Ali, and orders o2 and o3 receive free delivery.

In [13]:
%%writefile manual_knights.pl
both_knaves(A, B) :- A == knave, B == knave.
solve(A, B) :-
    member(A, [knight, knave]), member(B, [knight, knave]),
    (A == knight -> both_knaves(A, B) ; \+ both_knaves(A, B)).
at_least_one_knave(A, B) :- (A == knave ; B == knave).
solve_variant(A, B) :-
    member(A, [knight, knave]), member(B, [knight, knave]),
    (A == knight -> at_least_one_knave(A, B) ; \+ at_least_one_knave(A, B)).

Overwriting manual_knights.pl


In [14]:
run_prolog('manual_knights.pl', [
    'solve(A, B).',
    'solve_variant(A, B).',
])

?- solve(A, B).
['A'=knave,'B'=knight].

?- solve_variant(A, B).
['A'=knight,'B'=knave].



For “we are both knaves,” A must be a knave and B a knight: A’s statement is false. For “at least one of us is a knave,” A must be a knight and B a knave. A knave would make that statement true, contradicting the lying condition.

### Task 1: Printer Troubleshooting Expert System

In [15]:
printer_rules_task1 = [
    {"conditions": ["printer_offline", "printer_power_disconnected"],
     "conclusion": "printer_power_issue"},
    {"conditions": ["printer_offline", "printer_power_connected",
                    "printer_network_disconnected"],
     "conclusion": "printer_network_issue"}
]
it_rules = laptop_rules + printer_rules_task1
it_recommendations = {
    **laptop_recommendations,
    "printer_power_issue": "Check the printer power cable and switch on the printer.",
    "printer_network_issue": "Reconnect the printer to the network and check its settings."
}
facts = laptop_facts | {"printer_offline", "printer_power_disconnected"}
print("Scenario 1: laptop IP issue and printer power issue")
result = display_result(facts, it_rules, it_recommendations)
print("\nScenario 2: printer network issue")
result = display_result({"printer_offline", "printer_power_connected",
                         "printer_network_disconnected"}, it_rules, it_recommendations)

Scenario 1: laptop IP issue and printer power issue
Initial facts: internet_not_working, invalid_ip, other_devices_online, printer_offline, printer_power_disconnected, wifi_connected
New fact derived: wifi_without_internet
New fact derived: device_specific_problem
New fact derived: ip_configuration_issue
New fact derived: printer_power_issue
Newly derived facts: device_specific_problem, ip_configuration_issue, printer_power_issue, wifi_without_internet
Final Knowledge Base: device_specific_problem, internet_not_working, invalid_ip, ip_configuration_issue, other_devices_online, printer_offline, printer_power_disconnected, printer_power_issue, wifi_connected, wifi_without_internet
Possible Diagnoses and Solutions:
Diagnosis: ip_configuration_issue
Recommended Solution: Renew your IP address and reconnect to the network.
Diagnosis: device_specific_problem
Recommended Solution: Check the network settings of your laptop.
Diagnosis: wifi_without_internet
Recommended Solution: Check router an

Yes. The same forward_chaining() function diagnoses laptop and printer problems; only the facts, rules and recommendations change.

### Task 2: Smart Library Book Recommendation System

In [16]:
facts = {"interested_in_programming", "knows_basic_python",
         "interested_in_ai", "knows_basic_math"}
book_rules = [
    {"conditions": ["interested_in_programming", "knows_basic_python"],
     "conclusion": "ready_for_intermediate_programming"},
    {"conditions": ["interested_in_ai", "knows_basic_math"],
     "conclusion": "ready_for_introductory_ai"},
    {"conditions": ["ready_for_intermediate_programming"],
     "conclusion": "recommend_python_book"},
    {"conditions": ["ready_for_introductory_ai"],
     "conclusion": "recommend_ai_book"}
]
book_recommendations = {
    "recommend_python_book": "Python Crash Course — Eric Matthes",
    "recommend_ai_book": "Artificial Intelligence: A Modern Approach — Russell and Norvig"
}

def test_student(profile_name, student_facts):
    print("\nStudent:", profile_name)
    print("Initial Student Facts:", ", ".join(sorted(student_facts)))
    result = forward_chaining(student_facts.copy(), book_rules)
    print("Newly Derived Facts:", ", ".join(sorted(result - student_facts)) or "None")
    print("Final Knowledge Base:", ", ".join(sorted(result)))
    print("Recommended Books:")
    books = [book for fact, book in book_recommendations.items() if fact in result]
    print("\n".join(books) if books else "No suitable book recommendation found.")
    return result

result = test_student("Sample student: both interests", facts)


Student: Sample student: both interests
Initial Student Facts: interested_in_ai, interested_in_programming, knows_basic_math, knows_basic_python
New fact derived: ready_for_intermediate_programming
New fact derived: ready_for_introductory_ai
New fact derived: recommend_python_book
New fact derived: recommend_ai_book
Newly Derived Facts: ready_for_intermediate_programming, ready_for_introductory_ai, recommend_ai_book, recommend_python_book
Final Knowledge Base: interested_in_ai, interested_in_programming, knows_basic_math, knows_basic_python, ready_for_intermediate_programming, ready_for_introductory_ai, recommend_ai_book, recommend_python_book
Recommended Books:
Python Crash Course — Eric Matthes
Artificial Intelligence: A Modern Approach — Russell and Norvig


#### Testing two different student profiles

In [17]:
student1 = {"interested_in_programming", "knows_basic_python"}
student2 = {"interested_in_ai", "knows_basic_math"}
result = test_student("Student 1: Programming", student1)
result = test_student("Student 2: AI", student2)


Student: Student 1: Programming
Initial Student Facts: interested_in_programming, knows_basic_python
New fact derived: ready_for_intermediate_programming
New fact derived: recommend_python_book
Newly Derived Facts: ready_for_intermediate_programming, recommend_python_book
Final Knowledge Base: interested_in_programming, knows_basic_python, ready_for_intermediate_programming, recommend_python_book
Recommended Books:
Python Crash Course — Eric Matthes

Student: Student 2: AI
Initial Student Facts: interested_in_ai, knows_basic_math
New fact derived: ready_for_introductory_ai
New fact derived: recommend_ai_book
Newly Derived Facts: ready_for_introductory_ai, recommend_ai_book
Final Knowledge Base: interested_in_ai, knows_basic_math, ready_for_introductory_ai, recommend_ai_book
Recommended Books:
Artificial Intelligence: A Modern Approach — Russell and Norvig


The programming profile produces the Python book, the AI profile produces the AI book, and the sample profile satisfies both rule chains and receives both recommendations.

### Task 3: Campus Printer Troubleshooting Expert System

In [18]:
campus_printer_rules = [
    {"conditions": ["printer_not_working", "printer_power_disconnected"],
     "conclusion": "printer_power_issue"},
    {"conditions": ["printer_offline", "printer_power_connected",
                    "printer_network_disconnected"],
     "conclusion": "printer_network_issue"},
    {"conditions": ["printer_not_working", "paper_jammed"],
     "conclusion": "paper_jam_issue"},
    {"conditions": ["printer_not_working", "paper_not_available"],
     "conclusion": "paper_shortage_issue"}
]
campus_rules = laptop_rules + campus_printer_rules
campus_recommendations = {
    **it_recommendations,
    "paper_jam_issue": "Open the printer carefully and remove the jammed paper.",
    "paper_shortage_issue": "Load paper into the printer tray."
}

def printer_support():
    facts = set()
    questions = [
        ("Is the printer working?", None, "printer_not_working"),
        ("Is the printer offline?", "printer_offline", None),
        ("Is printer power connected?", "printer_power_connected", "printer_power_disconnected"),
        ("Is the network connection available?", "printer_network_available", "printer_network_disconnected"),
        ("Is paper available?", "paper_available", "paper_not_available"),
        ("Is paper jammed?", "paper_jammed", None)
    ]
    for prompt, yes_fact, no_fact in questions:
        fact = yes_fact if ask_yes_no(prompt) else no_fact
        if fact is not None:
            facts.add(fact)
    return display_result(facts, campus_rules, campus_recommendations)

#### Testing the system with recorded answers to input()

In [19]:
print('Power issue')
result = demonstrate_inputs(printer_support, ['no', 'yes', 'no', 'yes', 'yes', 'no'])

Power issue
Is the printer working? (yes/no): no
Is the printer offline? (yes/no): yes
Is printer power connected? (yes/no): no
Is the network connection available? (yes/no): yes
Is paper available? (yes/no): yes
Is paper jammed? (yes/no): no
Initial facts: paper_available, printer_network_available, printer_not_working, printer_offline, printer_power_disconnected
New fact derived: printer_power_issue
Newly derived facts: printer_power_issue
Final Knowledge Base: paper_available, printer_network_available, printer_not_working, printer_offline, printer_power_disconnected, printer_power_issue
Possible Diagnoses and Solutions:
Diagnosis: printer_power_issue
Recommended Solution: Check the printer power cable and switch on the printer.


In [20]:
print('Paper jam')
result = demonstrate_inputs(printer_support, ['no', 'no', 'yes', 'yes', 'yes', 'yes'])

Paper jam
Is the printer working? (yes/no): no
Is the printer offline? (yes/no): no
Is printer power connected? (yes/no): yes
Is the network connection available? (yes/no): yes
Is paper available? (yes/no): yes
Is paper jammed? (yes/no): yes
Initial facts: paper_available, paper_jammed, printer_network_available, printer_not_working, printer_power_connected
New fact derived: paper_jam_issue
Newly derived facts: paper_jam_issue
Final Knowledge Base: paper_available, paper_jam_issue, paper_jammed, printer_network_available, printer_not_working, printer_power_connected
Possible Diagnoses and Solutions:
Diagnosis: paper_jam_issue
Recommended Solution: Open the printer carefully and remove the jammed paper.


In [21]:
print('No supported diagnosis')
result = demonstrate_inputs(printer_support, ['no', 'no', 'yes', 'yes', 'yes', 'no'])

No supported diagnosis
Is the printer working? (yes/no): no
Is the printer offline? (yes/no): no
Is printer power connected? (yes/no): yes
Is the network connection available? (yes/no): yes
Is paper available? (yes/no): yes
Is paper jammed? (yes/no): no
Initial facts: paper_available, printer_network_available, printer_not_working, printer_power_connected
Newly derived facts: None
Final Knowledge Base: paper_available, printer_network_available, printer_not_working, printer_power_connected
Possible Diagnoses and Solutions:
No supported diagnosis found.


The three template scenarios identify a power issue, a paper jam and no supported diagnosis. The existing inference engine is reused, and the laptop rules remain in the knowledge base.

### Task 4: University Course Prerequisite Advisor (Prolog)

In [22]:
%%writefile course_advisor.pl
:- dynamic completed/2, prerequisite/2.
completed(student1, programming).
completed(student1, database).
completed(student2, programming).
completed(student3, mathematics).
prerequisite(data_science, programming).
prerequisite(data_science, mathematics).
prerequisite(artificial_intelligence, programming).
prerequisite(artificial_intelligence, mathematics).
prerequisite(web_engineering, programming).
student(S) :- setof(X, C^completed(X, C), Students), member(S, Students).
course(C) :- setof(X, P^prerequisite(X, P), Courses), member(C, Courses).
has_completed_prerequisite(Student, Course) :- completed(Student, Course).
eligible(Student, Course) :-
    student(Student), course(Course),
    forall(prerequisite(Course, Required),
           has_completed_prerequisite(Student, Required)).

Overwriting course_advisor.pl


In [23]:
run_prolog('course_advisor.pl', [
    'eligible(student1, data_science).',
    'eligible(student2, data_science).',
    'eligible(student3, artificial_intelligence).',
    'eligible(Student, web_engineering).',
    'has_completed_prerequisite(student1, programming).',
])

?- eligible(student1, data_science).
false.

?- eligible(student2, data_science).
false.

?- eligible(student3, artificial_intelligence).
false.

?- eligible(Student, web_engineering).
['Student'=student1].
['Student'=student2].

?- has_completed_prerequisite(student1, programming).
true.



In [24]:
run_prolog("course_advisor.pl", [
    "assertz(completed(student4, programming)), assertz(completed(student4, mathematics)), assertz(completed(student4, database)).",
    "eligible(student4, data_science).",
    "eligible(student4, artificial_intelligence).",
    "eligible(Student, web_engineering).",
    "assertz(prerequisite(web_engineering, database)), assertz(prerequisite(data_science, database)).",
    "eligible(Student, web_engineering).",
    "eligible(student2, web_engineering).",
    "eligible(student4, data_science)."
])

?- assertz(completed(student4, programming)), assertz(completed(student4, mathematics)), assertz(completed(student4, database)).
true.

?- eligible(student4, data_science).
true.

?- eligible(student4, artificial_intelligence).
true.

?- eligible(Student, web_engineering).
['Student'=student1].
['Student'=student2].
['Student'=student4].

?- assertz(prerequisite(web_engineering, database)), assertz(prerequisite(data_science, database)).
true.

?- eligible(Student, web_engineering).
['Student'=student1].
['Student'=student4].

?- eligible(student2, web_engineering).
false.

?- eligible(student4, data_science).
true.



The first three required eligibility queries fail because each student is missing a required course. Initially student1 and student2 qualify for web engineering. The new student4 has programming, mathematics and database. Adding database as a prerequisite for web engineering and data science removes student2’s web eligibility; student1 and student4 remain eligible. `forall` checks every prerequisite after the student and course are bound, so a variable query does not incorrectly combine different students’ completed courses.

### Task 5: Family Relationship Knowledge Base (Prolog)
Basic information consists of names, gender and parent-child facts. Ali and Fatima form generation 1; Ahmed, Sara, Omar and Hina form generation 2; Usman, Ayesha, Hamza and Zain form generation 3.

In [25]:
%%writefile family_relationships.pl
:- dynamic male/1, female/1, parent/2.
male(ali). male(ahmed). male(omar).
male(usman). male(hamza). male(zain).
female(fatima). female(sara). female(hina). female(ayesha).
parent(ali, ahmed). parent(fatima, ahmed).
parent(ali, sara). parent(fatima, sara).
parent(ahmed, usman). parent(hina, usman).
parent(ahmed, ayesha). parent(hina, ayesha).
parent(omar, hamza). parent(sara, hamza).
parent(omar, zain). parent(sara, zain).
father(X, Y) :- male(X), parent(X, Y).
mother(X, Y) :- female(X), parent(X, Y).
grandparent(X, Z) :- parent(X, Y), parent(Y, Z).
sibling(X, Y) :- parent(P, X), parent(P, Y), X \== Y.
brother(X, Y) :- male(X), sibling(X, Y).
sister(X, Y) :- female(X), sibling(X, Y).

Overwriting family_relationships.pl


In [26]:
run_prolog('family_relationships.pl', [
    'father(Who, usman).',
    'mother(Who, hamza).',
    'grandparent(ali, usman).',
    'sibling(usman, ayesha).',
    'brother(hamza, zain).',
    'setof(Who, sister(Who, usman), Sisters).',
    'setof(Child, parent(ali, Child), Children).',
])

?- father(Who, usman).
['Who'=ahmed].

?- mother(Who, hamza).
['Who'=sara].

?- grandparent(ali, usman).
true.

?- sibling(usman, ayesha).
true.

?- brother(hamza, zain).
true.

?- setof(Who, sister(Who, usman), Sisters).
['Who'=_11608,'Sisters'=[ayesha]].

?- setof(Child, parent(ali, Child), Children).
['Child'=_11746,'Children'=[ahmed,sara]].



In [27]:
run_prolog("family_relationships.pl", [
    "assertz(female(maryam)), assertz(male(bilal)), assertz(parent(ahmed, maryam)), assertz(parent(hina, maryam)), assertz(parent(omar, bilal)), assertz(parent(sara, bilal)).",
    "father(Who, maryam).",
    "mother(Who, bilal).",
    "grandparent(ali, maryam).",
    "setof(Who, sister(Who, usman), Sisters).",
    "brother(bilal, hamza).",
    "setof(Who, sibling(Who, zain), Siblings)."
])

?- assertz(female(maryam)), assertz(male(bilal)), assertz(parent(ahmed, maryam)), assertz(parent(hina, maryam)), assertz(parent(omar, bilal)), assertz(parent(sara, bilal)).
true.

?- father(Who, maryam).
['Who'=ahmed].

?- mother(Who, bilal).
['Who'=sara].

?- grandparent(ali, maryam).
true.

?- setof(Who, sister(Who, usman), Sisters).
['Who'=_11616,'Sisters'=[ayesha,maryam]].

?- brother(bilal, hamza).
true.

?- setof(Who, sibling(Who, zain), Siblings).
['Who'=_11826,'Siblings'=[bilal,hamza]].



#### Reasoning answers
1. Derived relationships do not need separate facts because general rules infer them from gender and parent information. This avoids redundant facts and inconsistent updates.
2. Prolog finds a shared parent P for X and Y and checks that X and Y are different individuals. Sharing either parent is sufficient. Two shared parents can produce duplicate proofs; `setof` gives distinct answers when listing relatives.
3. Adding Maryam and Bilal requires only new gender and parent facts. The same rules immediately infer their parents, grandparents and siblings because the rules use variables rather than particular names.